# Baseline Retrieval Hybrid dan Reranking Gemma

Notebook ini menjalankan dua eksperimen berurutan: retrieval hybrid Qdrant dense-sparse dan reranking kandidat menggunakan Gemma. Sel terakhir membuat resume Markdown dari artifact kedua eksperimen.

Prasyarat: akses Qdrant, koneksi internet, dan TOGETHER_API_KEY pada file .env.

In [5]:
from pathlib import Path
import json
import os
import subprocess
import sys

repo_root = Path.cwd()
if repo_root.name.lower() == "riset":
    repo_root = repo_root.parent
elif not (repo_root / "scripts" / "baseline_experiment.py").exists():
    candidates = [path for path in [Path.cwd(), *Path.cwd().parents] if (path / "scripts" / "baseline_experiment.py").exists()]
    if not candidates:
        raise FileNotFoundError("Tidak dapat menemukan root repository CDE-Mapper.")
    repo_root = candidates[0]

os.chdir(repo_root)
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")
os.environ.setdefault("QDRANT_TIMEOUT", "600")
print(f"Working directory: {repo_root}")
print(f"Python executable: {sys.executable}")

Working directory: d:\Program\cde_mapper_anie
Python executable: c:\Users\Rajha\miniconda3\envs\cde-mapper-py310\python.exe


In [6]:
# Ganti RUN_ID untuk menyimpan eksperimen baru tanpa menimpa artifact lama.
RUN_ID = "baseline-gemma3n-stage0"
RUN_RETRIEVAL = True
RUN_RERANKING = True

CONFIG_PATH = repo_root / "configs" / "baseline.yaml"
OUTPUT_DIR = repo_root / "data" / "output" / "baseline"
RETRIEVAL_ARTIFACT = OUTPUT_DIR / f"{RUN_ID}.jsonl"
MANIFEST_ARTIFACT = OUTPUT_DIR / f"{RUN_ID}.manifest.json"
RERANKED_ARTIFACT = OUTPUT_DIR / f"{RUN_ID}-reranked.jsonl"
HYBRID_METRICS_PATH = repo_root / "Riset" / f"{RUN_ID}-hybrid-metrics.json"
RERANKED_METRICS_PATH = repo_root / "Riset" / f"{RUN_ID}-reranked-metrics.json"
RESUME_PATH = repo_root / "Riset" / f"{RUN_ID}-experiment-resume.md"

required = [CONFIG_PATH, repo_root / "scripts" / "baseline_experiment.py", repo_root / "scripts" / "baseline_llm_rerank.py"]
missing = [path for path in required if not path.exists()]
if missing:
    raise FileNotFoundError("File wajib tidak ditemukan: " + ", ".join(map(str, missing)))
dotenv_path = repo_root / ".env"
if RUN_RERANKING and (not dotenv_path.exists() or not any(line.startswith(("TOGETHER_API_KEY=", "TOGATHER_API_KEY=")) for line in dotenv_path.read_text(encoding="utf-8").splitlines())):
    raise RuntimeError("Tambahkan TOGETHER_API_KEY pada .env sebelum menjalankan reranking.")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Run ID: {RUN_ID}")

Run ID: baseline-gemma3n-stage0


In [7]:
def run_command(command: list[str], allowed_returncodes: tuple[int, ...] = (0,)) -> int:
    """Jalankan command; hasil parsial dapat diberi return code yang diizinkan."""
    print("Running:\n" + " ".join(command))
    result = subprocess.run(command, cwd=repo_root, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout)
    if result.returncode not in allowed_returncodes:
        if result.stderr:
            print(result.stderr, file=sys.stderr)
        raise RuntimeError(f"Command gagal dengan exit code {result.returncode}.")
    if result.stderr:
        print(result.stderr, file=sys.stderr)
    if result.returncode:
        print(f"Command selesai dengan status parsial (exit code {result.returncode}).")
    return result.returncode

## Eksperimen 1: retrieval hybrid Qdrant dense-sparse

Menghasilkan kandidat Top-10, latency per sumber, error per query, dan manifest. LLM check dilewati agar retrieval dan reranking tetap terisolasi.

In [8]:
if RUN_RETRIEVAL:
    run_command([
        sys.executable, "scripts/baseline_experiment.py",
        "--config", str(CONFIG_PATH.relative_to(repo_root)),
        "--run-id", RUN_ID, "--skip-llm-check",
    ], allowed_returncodes=(0, 1))
if not RETRIEVAL_ARTIFACT.exists() or not MANIFEST_ARTIFACT.exists():
    raise FileNotFoundError("Artifact retrieval atau manifest tidak dihasilkan.")

Running:
c:\Users\Rajha\miniconda3\envs\cde-mapper-py310\python.exe scripts/baseline_experiment.py --config configs\baseline.yaml --run-id baseline-gemma3n-stage0 --skip-llm-check
Paring Text for Embedding=Patient hospital number
Paring Text for Embedding=Baseline time
Paring Text for Embedding=Age
Paring Text for Embedding=Sex assigned at birth
Paring Text for Embedding=Body weight
Paring Text for Embedding=Body height
Paring Text for Embedding=Waist circumference
Paring Text for Embedding=Body mass index (BMI) [Ratio]
Paring Text for Embedding=kilogram
Paring Text for Embedding=centimeter
{
  "run_id": "baseline-gemma3n-stage0",
  "status": "completed_with_errors",
  "started_at": "2026-07-13T06:17:13.494799+00:00",
  "completed_at": "2026-07-13T06:17:44.676272+00:00",
  "python": "3.10.20 | packaged by Anaconda, Inc. | (main, Jun 11 2026, 15:13:20) [MSC v.1942 64 bit (AMD64)]",
  "platform": "Windows-10-10.0.26200-SP0",
  "git_commit": "7ceb82f2ad740f448d4887a8910631102028437d",
  "


Embedding Entities and Synonyms: 100%|██████████| 1/1 [00:00<00:00,  3.35batch/s]

Embedding Entities and Synonyms: 100%|██████████| 1/1 [00:00<00:00,  9.05batch/s]

Embedding Entities and Synonyms: 100%|██████████| 1/1 [00:00<00:00,  9.16batch/s]

Embedding Entities and Synonyms: 100%|██████████| 1/1 [00:00<00:00,  7.20batch/s]

Embedding Entities and Synonyms: 100%|██████████| 1/1 [00:00<00:00,  7.73batch/s]

Embedding Entities and Synonyms: 100%|██████████| 1/1 [00:00<00:00,  8.43batch/s]

Embedding Entities and Synonyms: 100%|██████████| 1/1 [00:00<00:00,  8.45batch/s]

Embedding Entities and Synonyms: 100%|██████████| 1/1 [00:00<00:00,  9.12batch/s]

Embedding Entities and Synonyms: 100%|██████████| 1/1 [00:00<00:00,  7.79batch/s]

Embedding Entities and Synonyms: 100%|██████████| 1/1 [00:00<00:00,  8.98batch/s]

Fetching 6 files: 100%|██████████| 6/6 [00:00<00:00, 5753.50it/s]



## Eksperimen 2: reranking Gemma

Memakai artifact retrieval yang tersimpan dan hanya mengubah urutan kandidat. Artifact retrieval asli tidak diubah.

In [9]:
if RUN_RERANKING:
    run_command([
        sys.executable, "scripts/baseline_llm_rerank.py",
        "--config", str(CONFIG_PATH.relative_to(repo_root)),
        "--input", str(RETRIEVAL_ARTIFACT.relative_to(repo_root)),
        "--output", str(RERANKED_ARTIFACT.relative_to(repo_root)),
    ], allowed_returncodes=(0, 1))
if not RERANKED_ARTIFACT.exists():
    raise FileNotFoundError("Artifact reranking tidak dihasilkan.")

Running:
c:\Users\Rajha\miniconda3\envs\cde-mapper-py310\python.exe scripts/baseline_llm_rerank.py --config configs\baseline.yaml --input data\output\baseline\baseline-gemma3n-stage0.jsonl --output data\output\baseline\baseline-gemma3n-stage0-reranked.jsonl
LLMManager: Getting instance for key: google/gemma-3n-E4B-it
{
  "model": "google/gemma-3n-E4B-it",
  "query_count": 10,
  "failures": 0,
  "output": "data\\output\\baseline\\baseline-gemma3n-stage0-reranked.jsonl"
}



## Metrik dan perbandingan

Metrik dihitung ulang dari artifact JSONL dan gold identifier, bukan dari angka hard-code pada laporan.

In [10]:
import pandas as pd
from evaluation.baseline_retrieval_eval import evaluate, read_jsonl

hybrid_rows = read_jsonl(RETRIEVAL_ARTIFACT)
reranked_rows = read_jsonl(RERANKED_ARTIFACT)
hybrid_metrics = evaluate(hybrid_rows)
reranked_metrics = evaluate(reranked_rows)
HYBRID_METRICS_PATH.write_text(json.dumps({"input": str(RETRIEVAL_ARTIFACT), **hybrid_metrics}, indent=2), encoding="utf-8")
RERANKED_METRICS_PATH.write_text(json.dumps({"input": str(RERANKED_ARTIFACT), **reranked_metrics}, indent=2), encoding="utf-8")

keys = ["coverage", "accuracy@1", "accuracy@3", "accuracy@5", "accuracy@10", "precision@1", "precision@3", "precision@5", "precision@10", "recall@1", "recall@3", "recall@5", "recall@10", "mrr", "ndcg@10"]
comparison = pd.DataFrame({
    "Metrik": keys,
    "Hybrid Qdrant dense-sparse": [hybrid_metrics[key] for key in keys],
    "Setelah reranking Gemma": [reranked_metrics[key] for key in keys],
})
comparison["Delta reranking - hybrid"] = comparison["Setelah reranking Gemma"] - comparison["Hybrid Qdrant dense-sparse"]
latency = pd.DataFrame({
    "Metrik latency (ms)": ["mean", "p50", "p95", "max"],
    "Hybrid Qdrant dense-sparse": [hybrid_metrics["latency_ms"][key] for key in ("mean", "p50", "p95", "max")],
    "Setelah reranking Gemma": [reranked_metrics["latency_ms"][key] for key in ("mean", "p50", "p95", "max")],
})
display(comparison.style.format({"Hybrid Qdrant dense-sparse": "{:.4f}", "Setelah reranking Gemma": "{:.4f}", "Delta reranking - hybrid": "{:+.4f}"}))
display(latency.style.format({"Hybrid Qdrant dense-sparse": "{:.2f}", "Setelah reranking Gemma": "{:.2f}"}))

,Metrik,Hybrid Qdrant dense-sparse,Setelah reranking Gemma,Delta reranking - hybrid
0,coverage,1.0000,1.0000,+0.0000
1,accuracy@1,0.6000,0.5000,-0.1000
2,accuracy@3,0.6000,0.7000,+0.1000
3,accuracy@5,0.7000,0.8000,+0.1000
4,accuracy@10,0.9000,0.9000,+0.0000
5,precision@1,0.6000,0.5000,-0.1000
6,precision@3,0.2000,0.2333,+0.0333
7,precision@5,0.1400,0.1600,+0.0200
8,precision@10,0.0900,0.0900,+0.0000
9,recall@1,0.6000,0.5000,-0.1000


,Metrik latency (ms),Hybrid Qdrant dense-sparse,Setelah reranking Gemma
0,mean,1471.09,4220.73
1,p50,1430.44,3888.83
2,p95,1656.60,6573.89
3,max,1815.20,7559.44


## Resume hasil dua eksperimen

Sel ini membuat dan menyimpan resume per RUN_ID dari manifest, hasil retrieval, dan hasil reranking.

In [11]:
from IPython.display import Markdown, display

manifest = json.loads(MANIFEST_ARTIFACT.read_text(encoding="utf-8"))
rerank_success = sum(row.get("llm_rerank", {}).get("status") == "success" for row in reranked_rows)
summary = f"""# Resume Eksperimen Baseline: {RUN_ID}

## Retrieval hybrid Qdrant dense-sparse
- Status: {manifest['status']}
- Query: {hybrid_metrics['query_count']}; coverage: {hybrid_metrics['coverage']:.2%}
- Collection: {manifest['retrieval']['qdrant_collection']}
- Accuracy@1/@3/@5/@10: {hybrid_metrics['accuracy@1']:.2f} / {hybrid_metrics['accuracy@3']:.2f} / {hybrid_metrics['accuracy@5']:.2f} / {hybrid_metrics['accuracy@10']:.2f}
- MRR: {hybrid_metrics['mrr']:.4f}; NDCG@10: {hybrid_metrics['ndcg@10']:.4f}
- Latency mean/p50/p95: {hybrid_metrics['latency_ms']['mean']:.2f} / {hybrid_metrics['latency_ms']['p50']:.2f} / {hybrid_metrics['latency_ms']['p95']:.2f} ms

## Reranking Gemma
- Model: {reranked_rows[0].get('llm_rerank', {}).get('model', 'unknown')}
- Berhasil: {rerank_success}/{len(reranked_rows)} query
- Accuracy@1/@3/@5/@10: {reranked_metrics['accuracy@1']:.2f} / {reranked_metrics['accuracy@3']:.2f} / {reranked_metrics['accuracy@5']:.2f} / {reranked_metrics['accuracy@10']:.2f}
- MRR: {reranked_metrics['mrr']:.4f}; NDCG@10: {reranked_metrics['ndcg@10']:.4f}

## Delta reranking terhadap hybrid
- Accuracy@1: {reranked_metrics['accuracy@1'] - hybrid_metrics['accuracy@1']:+.4f}
- Accuracy@3: {reranked_metrics['accuracy@3'] - hybrid_metrics['accuracy@3']:+.4f}
- Accuracy@5: {reranked_metrics['accuracy@5'] - hybrid_metrics['accuracy@5']:+.4f}
- MRR: {reranked_metrics['mrr'] - hybrid_metrics['mrr']:+.4f}
- NDCG@10: {reranked_metrics['ndcg@10'] - hybrid_metrics['ndcg@10']:+.4f}

## Artifact
- Retrieval: {RETRIEVAL_ARTIFACT.relative_to(repo_root)}
- Manifest: {MANIFEST_ARTIFACT.relative_to(repo_root)}
- Reranking: {RERANKED_ARTIFACT.relative_to(repo_root)}
- Metrik hybrid: {HYBRID_METRICS_PATH.relative_to(repo_root)}
- Metrik reranking: {RERANKED_METRICS_PATH.relative_to(repo_root)}
"""
RESUME_PATH.write_text(summary, encoding="utf-8")
display(Markdown(summary))
print(f"Resume tersimpan: {RESUME_PATH}")

# Resume Eksperimen Baseline: baseline-gemma3n-stage0

## Retrieval hybrid Qdrant dense-sparse
- Status: completed_with_errors
- Query: 10; coverage: 100.00%
- Collection: concept_mapping_1
- Accuracy@1/@3/@5/@10: 0.60 / 0.60 / 0.70 / 0.90
- MRR: 0.6400; NDCG@10: 0.6965
- Latency mean/p50/p95: 1471.09 / 1430.44 / 1656.60 ms

## Reranking Gemma
- Model: google/gemma-3n-E4B-it
- Berhasil: 10/10 query
- Accuracy@1/@3/@5/@10: 0.50 / 0.70 / 0.80 / 0.90
- MRR: 0.6343; NDCG@10: 0.6982

## Delta reranking terhadap hybrid
- Accuracy@1: -0.1000
- Accuracy@3: +0.1000
- Accuracy@5: +0.1000
- MRR: -0.0057
- NDCG@10: +0.0017

## Artifact
- Retrieval: data\output\baseline\baseline-gemma3n-stage0.jsonl
- Manifest: data\output\baseline\baseline-gemma3n-stage0.manifest.json
- Reranking: data\output\baseline\baseline-gemma3n-stage0-reranked.jsonl
- Metrik hybrid: Riset\baseline-gemma3n-stage0-hybrid-metrics.json
- Metrik reranking: Riset\baseline-gemma3n-stage0-reranked-metrics.json


Resume tersimpan: d:\Program\cde_mapper_anie\Riset\baseline-gemma3n-stage0-experiment-resume.md
